In [2]:
import random
import os
from itertools import combinations
from collections import defaultdict

DATA_PATH = 'ml-100k/u.data'

print(f"Using dataset: {DATA_PATH}")

# Loading the data

print("Loading MovieLens 100k dataset...")
user_movies = defaultdict(set)

with open(DATA_PATH, 'r') as f:

    for line in f:
        parts = line.strip().split('\t')
        user_id  = int(parts[0])
        movie_id = int(parts[1])
        user_movies[user_id].add(movie_id)

num_users  = len(user_movies)
num_movies = len(set(mid for movies in user_movies.values() for mid in movies))

print(f"  Users  : {num_users}")
print(f"  Movies : {num_movies}")
print(f"  Ratings: {sum(len(v) for v in user_movies.values()):,}")

# EXACT JACCARD — all pairs

def jaccard(set_a, set_b):

    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

print("\nComputing exact Jaccard for all user pairs (this may take ~30s)...")
users = sorted(user_movies.keys())
pairs = list(combinations(users, 2))
print(f"  Total pairs: {len(pairs):,}")

exact_jaccard = {}
exact_above   = set()   # pairs with exact J >= 0.5

for u, v in pairs:

    j = jaccard(user_movies[u], user_movies[v])
    exact_jaccard[(u, v)] = j
    if j >= 0.5:
        exact_above.add((u, v))

print(f"\n  Pairs with exact Jaccard >= 0.5: {len(exact_above)}")
print(f"\n  {'User A':>8} {'User B':>8} {'Exact J':>10}")
print("  " + "-" * 32)

for (u, v) in sorted(exact_above, key=lambda x: -exact_jaccard[x]):
    print(f"  {u:>8} {v:>8} {exact_jaccard[(u,v)]:>10.6f}")

# MIN-HASH helpers

M = 1_000_003   # prime > num_movies, defines hash range [m]
P = 2**31 - 1   # large Mersenne prime for parameter space

def make_hash_params(t, seed=42):
    # Return t (a, b) pairs for hash functions
    random.seed(seed)
    return [(random.randint(1, P-1), random.randint(0, P-1))
            for _ in range(t)]

def compute_signatures(user_movies_dict, params):
    """
    Min-hash signature for every user.
    sig[user][i] = min_{movie in user_movies} h_i(movie)
    """
    sigs = {}
    for user, movies in user_movies_dict.items():
        sigs[user] = [min((a * mid + b) % M for mid in movies)
                      for a, b in params]
    return sigs

def approx_j(sig_a, sig_b):
    return sum(a == b for a, b in zip(sig_a, sig_b)) / len(sig_a)


# EXPERIMENTS: t = 50, 100, 200  |  5 runs each

T_VALUES  = [50, 100, 200]
NUM_RUNS  = 5
THRESHOLD = 0.5

print("\n" + "=" * 68)
print("MIN-HASH EXPERIMENTS  (t = 50 / 100 / 200,  5 runs each)")
print("=" * 68)

summary = {}

for t in T_VALUES:

    print(f"\n{'─'*68}")
    print(f"  t = {t} hash functions")
    print(f"{'─'*68}")

    run_fps, run_fns = [], []
    first_run_pairs  = None   # save run-1 details for display

    for run in range(NUM_RUNS):

        seed   = run * 97 + t          # distinct seed per (t, run)
        params = make_hash_params(t, seed=seed)
        sigs   = compute_signatures(user_movies, params)

        approx_above = set()
        approx_j_vals = {}
        for u, v in pairs:
            aj = approx_j(sigs[u], sigs[v])
            if aj >= THRESHOLD:
                approx_above.add((u, v))
                approx_j_vals[(u, v)] = aj

        fp = approx_above - exact_above   # predicted ≥ 0.5, exact < 0.5
        fn = exact_above - approx_above   # exact ≥ 0.5, missed

        run_fps.append(len(fp))
        run_fns.append(len(fn))

        if run == 0:
            first_run_pairs = (approx_above, approx_j_vals, fp, fn)

        print(f"  Run {run+1}: {len(approx_above):>4} pairs predicted >=0.5  |  "
              f"FP = {len(fp):>3}   FN = {len(fn):>3}")

    avg_fp = sum(run_fps) / NUM_RUNS
    avg_fn = sum(run_fns) / NUM_RUNS
    summary[t] = {'avg_fp': avg_fp, 'avg_fn': avg_fn,
                  'all_fps': run_fps, 'all_fns': run_fns}

    print(f"\n  ┌─ Averages over {NUM_RUNS} runs ────────────────────┐")
    print(f"  │  Avg False Positives : {avg_fp:>6.2f}               │")
    print(f"  │  Avg False Negatives : {avg_fn:>6.2f}               │")
    print(f"  └──────────────────────────────────────────────┘")

    # Detail table from run 1 
    approx_above_r1, aj_vals_r1, fp_r1, fn_r1 = first_run_pairs
    print(f"\n  Predicted pairs >=0.5 (Run 1, t={t}):")
    print(f"  {'User A':>8} {'User B':>8} {'Approx J':>10} {'Exact J':>10} {'Label':>6}")
    print("  " + "-" * 48)
    for (u, v) in sorted(approx_above_r1, key=lambda x: -aj_vals_r1[x]):
        ej    = exact_jaccard[(u, v)]
        label = "TP" if (u, v) in exact_above else "FP"
        print(f"  {u:>8} {v:>8} {aj_vals_r1[(u,v)]:>10.4f} {ej:>10.4f} {label:>6}")

    if fn_r1:
        print(f"\n  False Negatives (Run 1, t={t}) — exact >=0.5 but MISSED:")
        print(f"  {'User A':>8} {'User B':>8} {'Exact J':>10}")
        print("  " + "-" * 32)
        for (u, v) in sorted(fn_r1, key=lambda x: -exact_jaccard[x]):
            print(f"  {u:>8} {v:>8} {exact_jaccard[(u,v)]:>10.4f}")

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 68)
print("FINAL SUMMARY")
print("=" * 68)
print(f"  Exact pairs with Jaccard >= 0.5 : {len(exact_above)}")
print(f"\n  {'t':>6}  {'Avg FP':>10}  {'Avg FN':>10}  {'FP per run':>20}  {'FN per run':>20}")
print("  " + "-" * 75)
for t in T_VALUES:
    fps = summary[t]['all_fps']
    fns = summary[t]['all_fns']
    print(f"  {t:>6}  {summary[t]['avg_fp']:>10.2f}  {summary[t]['avg_fn']:>10.2f}  "
          f"  {fps}  {fns}")


Using dataset: ml-100k/u.data
Loading MovieLens 100k dataset...
  Users  : 943
  Movies : 1682
  Ratings: 100,000

Computing exact Jaccard for all user pairs (this may take ~30s)...
  Total pairs: 444,153

  Pairs with exact Jaccard >= 0.5: 10

    User A   User B    Exact J
  --------------------------------
       408      898   0.838710
       328      788   0.672956
       489      587   0.629921
       600      826   0.545455
       451      489   0.533333
       674      879   0.521739
       554      764   0.517007
       197      826   0.512987
       800      879   0.500000
       197      600   0.500000

MIN-HASH EXPERIMENTS  (t = 50 / 100 / 200,  5 runs each)

────────────────────────────────────────────────────────────────────
  t = 50 hash functions
────────────────────────────────────────────────────────────────────
  Run 1:   79 pairs predicted >=0.5  |  FP =  70   FN =   1
  Run 2:   49 pairs predicted >=0.5  |  FP =  42   FN =   3
  Run 3:  110 pairs predicted >=0.5  |